# Style-Bert-VITS2 (ver 2.7.0) のGoogle Colabでの学習

Google Colab上でStyle-Bert-VITS2の学習を行うことができます。

このnotebookでは、通常使用ではあなたのGoogle Driveにフォルダ`Style-Bert-VITS2`を作り、その内部での作業を行います。他のフォルダには触れません。
Google Driveを使わない場合は、初期設定のところで適切なパスを指定してください。

## 流れ

### 学習を最初からやりたいとき
上から順に実行していけばいいです。音声合成に必要なファイルはGoogle Driveの`Style-Bert-VITS2/model_assets/`に保存されます。また、途中経過も`Style-Bert-VITS2/Data/`に保存されるので、学習を中断したり、途中から再開することもできます。

### 学習を途中から再開したいとき
0と1を行い、3の前処理は飛ばして、4から始めてください。スタイル分け5は、学習が終わったら必要なら行ってください。


## 0. 環境構築

Style-Bert-VITS2の環境をcolab上に構築します。ランタイムがT4等のGPUバックエンドになっていることを確認し、実行してください。

**注意**: このセルを実行した後に「セッションがクラッシュしました」「不明な理由により、セッションがクラッシュしました。」等の警告が出ますが、**無視してそのまま先へ**進んでください。（一度ランタイムを再起動させてnumpy<2を強制させるため `exit()` を呼んでいることからの措置です。）

In [1]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/litagin02/Style-Bert-VITS2.git
%cd Style-Bert-VITS2/
!uv pip install --system -r requirements-colab.txt --no-progress
!python initialize.py --skip_default_models

exit()

downloading uv 0.11.14 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
fatal: destination path 'Style-Bert-VITS2' already exists and is not an empty directory.
/content/Style-Bert-VITS2
Using Python 3.12.13 environment at: /usr
Checked 26 packages in 101ms


In [2]:
# Google driveを使う方はこちらを実行してください。

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 1. 初期設定

学習とその結果を保存するディレクトリ名を指定します。
Google driveの場合はそのまま実行、カスタマイズしたい方は変更して実行してください。

In [3]:
# 展開先フォルダを作成
!mkdir -p /content/drive/MyDrive/Style-Bert-VITS2/Data

# mychar.zip を展開
!unzip -o /content/drive/MyDrive/mychar.zip -d /content/drive/MyDrive/Style-Bert-VITS2/Data/

# 展開結果を確認
!ls /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/
!ls /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/ | head -5
!wc -l /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/esd.list

Archive:  /content/drive/MyDrive/mychar.zip
   creating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/esd.list  
   creating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_001.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_002.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_003.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_004.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_005.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_006.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_007.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw/mychar_008.wav  
  inflating: /content/drive/MyDrive/Style-Bert-VITS2/Dat

In [4]:
# 作業ディレクトリを移動
%cd /content/Style-Bert-VITS2/

# 学習に必要なファイルや途中経過が保存されるディレクトリ
dataset_root = "/content/drive/MyDrive/Style-Bert-VITS2/Data"

# 学習結果（音声合成に必要なファイルたち）が保存されるディレクトリ
assets_root = "/content/drive/MyDrive/Style-Bert-VITS2/model_assets"

import yaml


with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

/content/Style-Bert-VITS2


## 2. 学習に使うデータ準備

すでに音声ファイル（1ファイル2-12秒程度）とその書き起こしデータがある場合は2.2を、ない場合は2.1を実行してください。

### 2.1 音声ファイルからのデータセットの作成（ある人はスキップ可）

音声ファイル（1ファイル2-12秒程度）とその書き起こしのデータセットを持っていない方は、（日本語の）音声ファイルのみから以下の手順でデータセットを作成することができます。Google drive上の`Style-Bert-VITS2/inputs/`フォルダに音声ファイル（wavやmp3等の通常の音声ファイル形式、1ファイルでも複数ファイルでも可）を置いて、下を実行すると、データセットが作られ、自動的に正しい場所へ配置されます。

**2024-06-02のVer 2.5以降**、`inputs/`フォルダにサブフォルダを2個以上作ってそこへ音声ファイルをスタイルに応じて振り分けて置くと、学習の際にサブディレクトリに応じたスタイルが自動的に作成されます。デフォルトスタイルのみでよい場合や手動でスタイルを後で作成する場合は`inputs/`直下へ入れれば大丈夫です。

`※ 以下はスキップでよいとのこと`

In [ ]:
# 元となる音声ファイル（wav形式）を入れるディレクトリ
input_dir = "/content/drive/MyDrive/Style-Bert-VITS2/inputs"
# モデル名（話者名）を入力
model_name = "your_model_name"

# こういうふうに書き起こして欲しいという例文（句読点の入れ方・笑い方や固有名詞等）
initial_prompt = "こんにちは。元気、ですかー？ふふっ、私は……ちゃんと元気だよ！"

!python slice.py -i {input_dir} --model_name {model_name}
!python transcribe.py --model_name {model_name} --initial_prompt {initial_prompt} --use_hf_whisper

成功したらそのまま3へ進んでください

### 2.2 音声ファイルと書き起こしデータがすでにある場合

指示に従って適切にデータセットを配置してください。

次のセルを実行して、学習データをいれるフォルダ（1で設定した`dataset_root`）を作成します。

In [5]:
import os

os.makedirs(dataset_root, exist_ok=True)

まず音声データと、書き起こしテキストを用意してください。

それを次のように配置します。
```
├── Data/
│   ├── {モデルの名前}
│   │   ├── esd.list
│   │   ├── raw/
│   │   │   ├── foo.wav
│   │   │   ├── bar.mp3
│   │   │   ├── style1/
│   │   │   │   ├── baz.wav
│   │   │   │   ├── qux.wav
│   │   │   ├── style2/
│   │   │   │   ├── corge.wav
│   │   │   │   ├── grault.wav
...
```

### 配置の仕方
- 上のように配置すると、`style1/`と`style2/`フォルダの内部（直下以外も含む）に入っている音声ファイルたちから、自動的にデフォルトスタイルに加えて`style1`と`style2`というスタイルが作成されます
- 特にスタイルを作る必要がない場合や、スタイル分類機能等でスタイルを作る場合は、`raw/`フォルダ直下に全てを配置してください。このように`raw/`のサブディレクトリの個数が0または1の場合は、スタイルはデフォルトスタイルのみが作成されます。
- 音声ファイルのフォーマットはwav形式以外にもmp3等の多くの音声ファイルに対応しています

### 書き起こしファイル`esd.list`

`Data/{モデルの名前}/esd.list` ファイルには、以下のフォーマットで各音声ファイルの情報を記述してください。


```
path/to/audio.wav(wavファイル以外でもこう書く)|{話者名}|{言語ID、ZHかJPかEN}|{書き起こしテキスト}
```

- ここで、最初の`path/to/audio.wav`は、`raw/`からの相対パスです。つまり、`raw/foo.wav`の場合は`foo.wav`、`raw/style1/bar.wav`の場合は`style1/bar.wav`となります。
- 拡張子がwavでない場合でも、`esd.list`には`wav`と書いてください、つまり、`raw/bar.mp3`の場合でも`bar.wav`と書いてください。


例：
```
foo.wav|hanako|JP|こんにちは、元気ですか？
bar.wav|taro|JP|はい、聞こえています……。何か用ですか？
style1/baz.wav|hanako|JP|今日はいい天気ですね。
style1/qux.wav|taro|JP|はい、そうですね。
...
english_teacher.wav|Mary|EN|How are you? I'm fine, thank you, and you?
...
```
もちろん日本語話者の単一話者データセットでも構いません。

## 3. 学習の前処理

次に学習の前処理を行います。必要なパラメータをここで指定します。次のセルに設定等を入力して実行してください。「～～かどうか」は`True`もしくは`False`を指定してください。

In [6]:
# 上でつけたフォルダの名前`Data/{model_name}/`
model_name = "mychar"

# JP-Extra （日本語特化版）を使うかどうか。日本語の能力が向上する代わりに英語と中国語は使えなくなります。
use_jp_extra = True

# 学習のバッチサイズ。VRAMのはみ出具合に応じて調整してください。
batch_size = 4

# 学習のエポック数（データセットを合計何周するか）。
# 100で多すぎるほどかもしれませんが、もっと多くやると質が上がるのかもしれません。
epochs = 100

# 保存頻度。何ステップごとにモデルを保存するか。分からなければデフォルトのままで。
save_every_steps = 1000

# 音声ファイルの音量を正規化するかどうか
normalize = False

# 音声ファイルの開始・終了にある無音区間を削除するかどうか
trim = False

# 読みのエラーが出た場合にどうするか。
# "raise"ならテキスト前処理が終わったら中断、"skip"なら読めない行は学習に使わない、"use"なら無理やり使う
yomi_error = "skip"

上のセルが実行されたら、次のセルを実行して学習の前処理を行います。

３つのセルを追記

In [21]:
# BertJapaneseTokenizer に必要な依存（MeCab系）
!pip install -q fugashi unidic-lite

import shutil

file_path = "/content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py"
backup_path = file_path + ".bak"

# バックアップ
shutil.copy(file_path, backup_path)
print(f"✓ バックアップ: {backup_path}")

# 読み込み
with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

# --- パッチ1: import に BertJapaneseTokenizer を追加 ---
old_import = """from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DebertaV2Model,
    DebertaV2TokenizerFast,
    PreTrainedModel,
    PreTrainedTokenizer,
    PreTrainedTokenizerFast,
)"""

new_import = """from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    BertJapaneseTokenizer,
    DebertaV2Model,
    DebertaV2TokenizerFast,
    PreTrainedModel,
    PreTrainedTokenizer,
    PreTrainedTokenizerFast,
)"""

assert old_import in content, "import部分が見つかりません"
content = content.replace(old_import, new_import)
print("✓ Patch 1: import に BertJapaneseTokenizer 追加")

# --- パッチ2: tokenizer ロード部分に JP 分岐を追加 ---
old_load = """    if language == Languages.EN:
        __loaded_tokenizers[language] = DebertaV2TokenizerFast.from_pretrained(
            pretrained_model_name_or_path,
            cache_dir=cache_dir,
            revision=revision,
        )
    else:
        __loaded_tokenizers[language] = AutoTokenizer.from_pretrained(
            pretrained_model_name_or_path,
            cache_dir=cache_dir,
            revision=revision,
            use_fast=True,  # デフォルトで True だが念のため明示的に指定
        )"""

new_load = """    if language == Languages.EN:
        __loaded_tokenizers[language] = DebertaV2TokenizerFast.from_pretrained(
            pretrained_model_name_or_path,
            cache_dir=cache_dir,
            revision=revision,
        )
    elif language == Languages.JP:
        __loaded_tokenizers[language] = BertJapaneseTokenizer.from_pretrained(
            pretrained_model_name_or_path,
            cache_dir=cache_dir,
        )
    else:
        __loaded_tokenizers[language] = AutoTokenizer.from_pretrained(
            pretrained_model_name_or_path,
            cache_dir=cache_dir,
            revision=revision,
            use_fast=True,  # デフォルトで True だが念のため明示的に指定
        )"""

assert old_load in content, "tokenizer ロード部分が見つかりません"
content = content.replace(old_load, new_load)
print("✓ Patch 2: JP分岐に BertJapaneseTokenizer 追加")

# 書き戻し
with open(file_path, "w", encoding="utf-8") as f:
    f.write(content)

print(f"\n✓ パッチ適用完了")
print(f"  失敗時の復元: !cp {backup_path} {file_path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 18.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 39.9 MB/s eta 0:00:00
✓ バックアップ: /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py.bak
✓ Patch 1: import に BertJapaneseTokenizer 追加
✓ Patch 2: JP分岐に BertJapaneseTokenizer 追加

✓ パッチ適用完了
  失敗時の復元: !cp /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py.bak /content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py


In [22]:
!rm -f /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/esd.list.cleaned
!rm -f /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/train.list
!rm -f /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/val.list
!rm -f /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/config.json
!rm -rf /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/wavs
!rm -rf /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/models
print("✓ クリーンアップ完了")

✓ クリーンアップ完了


In [24]:
import subprocess

# pyannote/audio/core/io.py の場所を特定
result = subprocess.run(
    ["find", "/usr/local/lib", "-name", "io.py", "-path", "*pyannote/audio/core*"],
    capture_output=True, text=True
)
io_path = result.stdout.strip().split("\n")[0]
print(f"対象ファイル: {io_path}")

# バックアップ
import shutil
shutil.copy(io_path, io_path + ".bak")
print(f"✓ バックアップ: {io_path}.bak")

# 修正前の該当行を確認
with open(io_path, "r") as f:
    content = f.read()

# パッチ1: AudioMetaData の型ヒントを object に置換
if "-> torchaudio.AudioMetaData:" in content:
    content = content.replace("-> torchaudio.AudioMetaData:", "-> object:")
    print("✓ Patch 1: AudioMetaData 型ヒント置換")
else:
    print("⚠ AudioMetaData 型ヒントが見つからない（既に修正済み？）")

# パッチ2: list_audio_backends も予防的に修正（Issue #613より）
if "torchaudio.list_audio_backends()" in content:
    content = content.replace(
        "torchaudio.list_audio_backends()",
        "getattr(torchaudio, 'list_audio_backends', lambda: ['sox_io'])()"
    )
    print("✓ Patch 2: list_audio_backends 予防的修正")

# 書き戻し
with open(io_path, "w") as f:
    f.write(content)

print(f"\n✓ pyannote/audio パッチ完了")

対象ファイル: /usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py
✓ バックアップ: /usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py.bak
✓ Patch 1: AudioMetaData 型ヒント置換
✓ Patch 2: list_audio_backends 予防的修正

✓ pyannote/audio パッチ完了


In [26]:
import shutil
import re

model_path = "/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py"

# バックアップ
shutil.copy(model_path, model_path + ".bak")
print(f"✓ バックアップ: {model_path}.bak")

# 読み込み
with open(model_path, "r") as f:
    content = f.read()

# 置換前のヒット数を確認
hit_count = content.count("use_auth_token=use_auth_token")
print(f"置換対象箇所: {hit_count} 箇所")

# パッチ: use_auth_token=use_auth_token → token=use_auth_token
new_content = content.replace(
    "use_auth_token=use_auth_token",
    "token=use_auth_token"
)

# 書き戻し
with open(model_path, "w") as f:
    f.write(new_content)

# 結果確認
with open(model_path, "r") as f:
    verify = f.read()
remaining = verify.count("use_auth_token=use_auth_token")
patched = verify.count("token=use_auth_token")
print(f"✓ パッチ後: 残存 {remaining}、置換済み {patched}")
print("✓ pyannote/audio/core/model.py パッチ完了")

✓ バックアップ: /usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py.bak
置換対象箇所: 2 箇所
✓ パッチ後: 残存 0、置換済み 2
✓ pyannote/audio/core/model.py パッチ完了


In [28]:
import shutil
import os

# ============================================================
# Patch A: lightning_fabric/utilities/cloud_io.py
# Qiita項目6: torch.load の weights_only デフォルト変更問題
# 今のStep 5エラーの原因
# ============================================================
cloud_io_path = "/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py"
shutil.copy(cloud_io_path, cloud_io_path + ".bak")

with open(cloud_io_path, "r") as f:
    content = f.read()

old_a = "weights_only=weights_only,"
new_a = "weights_only=False if weights_only is None else weights_only,"

count_a = content.count(old_a)
if count_a > 0 and new_a not in content:
    content = content.replace(old_a, new_a)
    with open(cloud_io_path, "w") as f:
        f.write(content)
    print(f"✓ Patch A: cloud_io.py 修正 ({count_a} 箇所)")
else:
    print(f"⚠ Patch A: スキップ (既に修正済み or 該当なし)")

# ============================================================
# Patch B: style_bert_vits2/models/models_jp_extra.py
# Qiita項目7: 推論時 fp16/fp32 型不一致（use_jp_extra=True の場合に使用）
# 学習完了後の音声合成で必要
# ============================================================
jp_extra_path = "/content/Style-Bert-VITS2/style_bert_vits2/models/models_jp_extra.py"
shutil.copy(jp_extra_path, jp_extra_path + ".bak")

with open(jp_extra_path, "r") as f:
    content = f.read()

old_b = "bert_emb = self.bert_proj(bert).transpose(1, 2)"
new_b = "bert_emb = self.bert_proj(bert.to(self.bert_proj.weight.dtype)).transpose(1, 2)"

if old_b in content and new_b not in content:
    content = content.replace(old_b, new_b)
    with open(jp_extra_path, "w") as f:
        f.write(content)
    print("✓ Patch B: models_jp_extra.py bert_proj 型キャスト追加")
else:
    print("⚠ Patch B: スキップ (既に修正済み or 該当なし)")

# ============================================================
# Patch C: style_bert_vits2/models/models.py
# Qiita項目7: 推論時 fp16/fp32 型不一致（多言語用、念のため）
# ============================================================
models_path = "/content/Style-Bert-VITS2/style_bert_vits2/models/models.py"
shutil.copy(models_path, models_path + ".bak")

with open(models_path, "r") as f:
    content = f.read()

replacements_c = [
    ("bert_emb = self.bert_proj(bert).transpose(1, 2)",
     "bert_emb = self.bert_proj(bert.to(self.bert_proj.weight.dtype)).transpose(1, 2)"),
    ("ja_bert_emb = self.ja_bert_proj(ja_bert).transpose(1, 2)",
     "ja_bert_emb = self.ja_bert_proj(ja_bert.to(self.ja_bert_proj.weight.dtype)).transpose(1, 2)"),
    ("en_bert_emb = self.en_bert_proj(en_bert).transpose(1, 2)",
     "en_bert_emb = self.en_bert_proj(en_bert.to(self.en_bert_proj.weight.dtype)).transpose(1, 2)"),
]

patched_c = 0
for old, new in replacements_c:
    if old in content and new not in content:
        content = content.replace(old, new)
        patched_c += 1

with open(models_path, "w") as f:
    f.write(content)
print(f"✓ Patch C: models.py bert_proj 型キャスト追加 ({patched_c}/3 箇所)")

print("\n=== 全パッチ適用完了 ===")
print("これでQiita記事掲載の主要バグは全て対処済み:")
print("  ✓ Patch (前): bert_models.py トークナイザー  → Step 4 解決済")
print("  ✓ Patch (前): pyannote io.py AudioMetaData  → Step 5前半 解決済")
print("  ✓ Patch (前): pyannote model.py use_auth_token → Step 5中盤 解決済")
print("  ✓ Patch A: lightning cloud_io.py weights_only → 今回")
print("  ✓ Patch B: models_jp_extra.py 型キャスト → 学習後の推論用")
print("  ✓ Patch C: models.py 型キャスト → 学習後の推論用")

✓ Patch A: cloud_io.py 修正 (3 箇所)
✓ Patch B: models_jp_extra.py bert_proj 型キャスト追加
✓ Patch C: models.py bert_proj 型キャスト追加 (3/3 箇所)

=== 全パッチ適用完了 ===
これでQiita記事掲載の主要バグは全て対処済み:
  ✓ Patch (前): bert_models.py トークナイザー  → Step 4 解決済
  ✓ Patch (前): pyannote io.py AudioMetaData  → Step 5前半 解決済
  ✓ Patch (前): pyannote model.py use_auth_token → Step 5中盤 解決済
  ✓ Patch A: lightning cloud_io.py weights_only → 今回
  ✓ Patch B: models_jp_extra.py 型キャスト → 学習後の推論用
  ✓ Patch C: models.py 型キャスト → 学習後の推論用


In [29]:
from gradio_tabs.train import preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker


pyopenjtalk_worker.initialize_worker()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=use_jp_extra,
    val_per_lang=0,
    log_interval=200,
    yomi_error=yomi_error,
)

05-16 09:35:07 |  INFO  | train.py:72 | Step 1: start initialization...
model_name: mychar, batch_size: 4, epochs: 100, save_every_steps: 1000, freeze_ZH_bert: False, freeze_JP_bert: False, freeze_EN_bert: False, freeze_style: False, freeze_decoder: False, use_jp_extra: True
05-16 09:35:07 |WARNING | train.py:103 | Step 1: /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/models already exists, so copy it to backup to /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/models_backup
05-16 09:35:14 |SUCCESS | train.py:132 | Step 1: initialization finished.
05-16 09:35:14 |  INFO  | train.py:137 | Step 2: start resampling...
05-16 09:35:14 |  INFO  | subprocess.py:23 | Running: resample.py -i /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw -o /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/wavs --num_processes 2 --sr 44100
05-16 09:35:26 |SUCCESS | subprocess.py:38 | Success: resample.py -i /content/drive/MyDrive/Style-Bert-VITS2/Data/mychar/raw -o /content/drive/MyDrive/St

(True, 'Success: 全ての前処理が完了しました。ターミナルを確認しておかしいところがないか確認するのをおすすめします。')

## 4. 学習

前処理が正常に終わったら、学習を行います。次のセルを実行すると学習が始まります。

学習の結果は、上で指定した`save_every_steps`の間隔で、Google Driveの中の`Style-Bert-VITS2/Data/{モデルの名前}/model_assets/`フォルダに保存されます。

このフォルダをダウンロードし、ローカルのStyle-Bert-VITS2の`model_assets`フォルダに上書きすれば、学習結果を使うことができます。

In [30]:
# 上でつけたモデル名を入力。学習を途中からする場合はきちんとモデルが保存されているフォルダ名を入力。
model_name = "mychar"


import yaml
from gradio_tabs.train import get_path

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

※　今回は「使う」

In [31]:
# 日本語特化版を「使う」場合
!python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

2026-05-16 09:36:51.821016: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
05-16 09:37:03 |  INFO  | train_ms_jp_extra.py:118 | Loading configuration from config 0
05-16 09:37:03 |  INFO  | train_ms_jp_extra.py:118 | Loading configuration from config localhost
05-16 09:37:03 |  INFO  | train_ms_jp_extra.py:118 | Loading configuration from config 10086
05-16 09:37:03 |  INFO  | train_ms_jp_extra.py:118 | Loading configuration from config 0
05-16 09:37:03 |  INFO  | train_ms_jp_extra.py:118 | Loading configuration from config 1
05-16 09:37:03 |  INFO  | train_ms_jp_extra.py:120 | Loading environment variables 
MASTER_ADDR: localhost,
MASTER_PORT: 10086,
WORLD_SIZE: 1,
RANK: 0,
LOCAL_RANK: 0
05-16 09:37:03 |  INFO  | default_style.py:54 | At least 

In [ ]:
# 日本語特化版を「使わない」場合
!python train_ms.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

In [ ]:
# 学習結果を試す・マージ・スタイル分けはこちらから
!python app.py --share

In [ ]:
# ONNX変換は、変換したいsafetensorsファイルを指定してこのセルを実行してください。
!python convert_onnx.py --model "Data/your_model/your_model_e100_s10000.safetensors"